# অধ্যায় ৭: চ্যাট অ্যাপ্লিকেশন তৈরি
## Azure OpenAI API দ্রুত শুরু


## ওভারভিউ
এই নোটবুকটি [Azure OpenAI স্যাম্পল রেপোজিটরি](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) থেকে অভিযোজিত, যা এমন নোটবুকগুলো অন্তর্ভুক্ত করে যেগুলো [OpenAI](notebook-openai.ipynb) সেবা অ্যাক্সেস করে।

পাইথন OpenAI API Azure OpenAI এর সাথেও কাজ করে, কিছু পরিবর্তনের সাথে। এখানে তার পার্থক্যগুলি সম্পর্কে আরো জানুন: [Python দিয়ে OpenAI এবং Azure OpenAI এন্ডপয়েন্টগুলোর মধ্যে কীভাবে পরিবর্তন করবেন](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

আরও দ্রুত শুরু করার উদাহরণের জন্য দয়া করে অফিসিয়াল [Azure OpenAI কুইকস্টার্ট ডকুমেন্টেশন](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst) দেখুন


## বিষয়বস্তু সূচি  

[ওভারভিউ](#overview)  
[Azure OpenAI সার্ভিস দিয়ে শুরু করা](#getting-started-with-azure-openai-service)  
[আপনার প্রথম প্রম্পট তৈরি করুন](#build-your-first-prompt)  

[ব্যবহার ক্ষেত্রসমূহ](#use-cases)    
[১. টেক্সট সংক্ষিপ্তকরণ](#summarize-text)  
[২. টেক্সট শ্রেণীবিভাগ](#classify-text)  
[৩. নতুন পণ্যের নাম তৈরি করুন](#generate-new-product-names)  
[৪. একটি শ্রেণীবিভাজক ফাইন টিউন করুন](#fine-tune-a-classifier)  
[৫. এম্বেডিংস](#embeddings)

[তথ্যসূত্র](#references)


### Azure OpenAI Service শুরু করা

নতুন গ্রাহকদের Azure OpenAI Service-এ প্রবেশাধিকার পেতে [অনুমতির জন্য আবেদন](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst) করতে হবে।  
অনুমোদন সম্পন্ন হওয়ার পরে, গ্রাহকরা Azure পোর্টালে লগ ইন করতে পারবেন, একটি Azure OpenAI Service রিসোর্স তৈরি করতে পারবেন, এবং স্টুডিওর মাধ্যমে মডেলগুলোর সাথে পরীক্ষা শুরু করতে পারবেন  

[দ্রুত শুরু করার জন্য দুর্দান্ত উৎস](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### আপনার প্রথম প্রম্পট তৈরি করুন  
এই সংক্ষিপ্ত অনুশীলনটি একটি সহজ কাজ "সারসংক্ষেপ" করার জন্য OpenAI মডেলে প্রম্পট জমা দেওয়ার একটি মৌলিক পরিচয় দেবে।  


**পদক্ষেপগুলি**:  
1. আপনার পাইথন পরিবেশে OpenAI লাইব্রেরি ইনস্টল করুন  
2. সাধারণ সহায়ক লাইব্রেরি লোড করুন এবং আপনার তৈরি OpenAI সার্ভিসের জন্য আপনার সাধারণ OpenAI সিকিউরিটি ক্রেডেনশিয়াল সেট করুন  
3. আপনার কাজের জন্য একটি মডেল নির্বাচন করুন  
4. মডেলের জন্য একটি সাধারণ প্রম্পট তৈরি করুন  
5. মডেল API-তে আপনার অনুরোধ জমা দিন!  


### 1. OpenAI ইনস্টল করুন


  > [!NOTE] এই ধাপটি প্রয়োজন নেই যদি এই নোটবুকটি Codespaces বা Devcontainer এর মধ্যে চালানো হয়


In [ ]:
%pip install openai python-dotenv

### ২। সহায়ক লাইব্রেরি আমদানি করুন এবং শংসাপত্র ইনস্ট্যানস করুন


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. সঠিক মডেল খোঁজা  
GPT-4o এবং GPT-4o মিনি মত মডেলগুলি প্রাকৃতিক ভাষা বুঝতে এবং তৈরি করতে পারে। Microsoft Foundry-তে Azure OpenAI বিভিন্ন মডেল ক্ষমতা প্রদান করে, প্রতিটির বিভিন্ন স্তরের ক্ষমতা এবং গতি থাকে যা বিভিন্ন কাজের জন্য উপযুক্ত। 

[Microsoft Foundry-তে Azure OpenAI এর মডেলগুলি](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## ৪. প্রম্পট ডিজাইন  

"বড় ভাষাগত মডেলগুলোর জাদু হল তারা বিশাল মাত্রার টেক্সটের ওপর এই পূর্বাভাস ভুলকে কমিয়ে এনে প্রশীক্ষিত হয়, ফলস্বরূপ মডেলগুলি এই পূর্বাভাসের জন্য দরকারী ধারণাগুলো শেখে। উদাহরণস্বরূপ, তারা নিম্নলিখিত ধারণাগুলো শেখে"(1):

* কীভাবে বানান করা হয়
* কীভাবে ব্যাকরণ কাজ করে
* কীভাবে প্যারাফ্রেজ করতে হয়
* কীভাবে প্রশ্নের উত্তর দিতে হয়
* কীভাবে কথোপকথন চালাতে হয়
* কীভাবে অনেক ভাষায় লিখতে হয়
* কীভাবে কোড লিখতে হয়
* ইত্যাদি।

#### কীভাবে একটি বড় ভাষাগত মডেল নিয়ন্ত্রণ করবেন  
"একটি বড় ভাষাগত মডেলের সব ইনপুটের মধ্যে সবচেয়ে প্রভাবশালী হলো টেক্সট প্রম্পট(1)।

বড় ভাষাগত মডেলগুলোকে কয়েকটি উপায়ে আউটপুট দেওয়ার প্রম্পট দেওয়া যেতে পারে:

- নির্দেশনা: মডেলকে বলুন আপনি কী চান
- সম্পূর্ণতা: মডেলকে আপনার চাওয়ার শুরুটা শেষ করতে প্ররোচিত করুন
- প্রদর্শনী: মডেলকে দেখান আপনি কী চান, নিম্নলিখিতের সাথে:
- প্রম্পটে কয়েকটি উদাহরণ
- একটি ফাইন-টিউনিং প্রশিক্ষণ ডেটাসেটের কয়েক শত বা হাজার উদাহরণ 



#### প্রম্পট তৈরির তিনটি মৌলিক নির্দেশিকা রয়েছে:

**দেখান এবং বলুন**। আপনি কী চান তা স্পষ্ট করুন, হয় নির্দেশনা, উদাহরণ বা দুয়ের সমন্বয়ের মাধ্যমে। যদি আপনি মডেলকে একটি তালিকাকে বর্ণানুক্রমিক ক্রমে সাজাতে চান অথবা একটি অনুচ্ছেদকে অনুভূতির ভিত্তিতে শ্রেণীবদ্ধ করতে চান, তাহলে মডেলকে দেখান যে সেটাই আপনি চান।

**উচ্চ মানের ডেটা প্রদান করুন**। আপনি যদি একটি শ্রেণীবিন্যাসক অর্থাৎ ক্লাসিফায়ার তৈরি করতে চান অথবা মডেলকে কোনো ধাঁচ অনুসরণ করাতে চান, তা হলে যথেষ্ট উদাহরণ থাকা উচিত। আপনার উদাহরণগুলো সংশোধন করা নিশ্চিত করুন — মডেল সাধারণত মৌলিক বানান ভুলগুলো বুঝে সাড়া দিতে পারে, তবে এটি মনে করতে পারে এটি অভিপ্রেত এবং এতে সাড়া প্রভাবিত হতে পারে।

**আপনার সেটিংস পরীক্ষা করুন।** তাপমাত্রা এবং top_p সেটিংস নিয়ন্ত্রণ করে মডেল কতটা সুনির্দিষ্টভাবে সাড়া তৈরি করে। আপনি যদি এমন একটি সাড়া চান যেখানে শুধুমাত্র একটাই সঠিক উত্তর থাকে, তাহলে এগুলো নীচুতে সেট করতে চান। আর আপনি যদি আরও বৈচিত্র্যময় সাড়া চান, তবে এগুলো উচ্চ স্থানে সেট করতে পারেন। সবচেয়ে বড় ভুল হলো এই সেটিংসগুলোকে "চতুরতা" বা "সৃজনশীলতা" নিয়ন্ত্রণ মনে করা।


উৎস: https://learn.microsoft.com/azure/ai-services/openai/overview


ছবি আপনার প্রথম টেক্সট প্রম্পট তৈরি করছে!


### ৫. জমা দিন!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### একই কলটি পুনরাবৃত্তি করুন, ফলাফলগুলি কীভাবে তুলনা করে? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## টেক্সট সারাংশ  
#### চ্যালেঞ্জ  
একটি টেক্সট অংশের শেষে 'tl;dr:' যোগ করে টেক্সটকে সংক্ষেপ করুন। লক্ষ্য করুন কিভাবে মডেল অতিরিক্ত নির্দেশনা ছাড়াই বিভিন্ন কাজ সম্পাদন করতে পারে। আপনি মডেলের আচরণ পরিবর্তন করতে এবং প্রাপ্ত সংক্ষেপণ কাস্টমাইজ করতে tl;dr ছাড়া আরও বর্ণনামূলক প্রম্পট নিয়ে পরীক্ষা-নিরীক্ষা করতে পারেন(3)।  

সাম্প্রতিক কাজগুলো দেখিয়েছে যে একটি বড় টেক্সট কর্পাসে প্রি-ট্রেনিং-এর মাধ্যমে এবং পরবর্তী একটি নির্দিষ্ট কাজের উপর ফাইন-টিউনিং করার মাধ্যমে অনেক NLP কাজ এবং বেঞ্চমার্কে উল্লেখযোগ্য উন্নতি হয়েছে। যদিও সাধারণত আর্কিটেকচারে টাস্ক-অজ্ঞাত হওয়া সত্ত্বেও, এই পদ্ধতিটি এখনও হাজার হাজার বা কয়েক হাজার উদাহরণসহ টাস্ক-নির্দিষ্ট ফাইন-টিউনিং ডেটাসেট প্রয়োজন। বিপরীতে, মানুষ সাধারণত কিছু উদাহরণ বা সরল নির্দেশনা থেকে একটি নতুন ভাষাগত কাজ করতে পারে - যা বর্তমান NLP সিস্টেমগুলো এখনও বেশিরভাগ সময় করতে সংগ্রাম করে। এখানে আমরা দেখিয়েছি যে ভাষা মডেলকে বড় করলে টাস্ক-অজ্ঞাত, কয়েক-শট পারফরম্যান্স বিশেষভাবে উন্নত হয়, এমনকি কখনও কখনও পূর্ববর্তী সর্বোচ্চ মানদণ্ডের ফাইন-টিউনিং পদ্ধতি সমকক্ষেও পৌঁছাতে পারে।  



টিএল;ডিআর  


# বিভিন্ন ব্যবহারের ক্ষেত্রে অনুশীলন  
1. টেক্সট সারসংক্ষেপ করুন  
2. টেক্সট শ্রেণীবদ্ধ করুন  
3. নতুন পণ্যের নাম তৈরি করুন  
4. এম্বেডিংস  
5. একটি শ্রেণীবিন্যাসকারী ফাইন টিউন করুন  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## পাঠ্য শ্রেণীবদ্ধ করুন  
#### চ্যালেঞ্জ  
আইটেমগুলি অনুমান করার সময় প্রদত্ত বিভাগে শ্রেণীবদ্ধ করুন। নিম্নলিখিত উদাহরণে, আমরা উভয় বিভাগ এবং শ্রেণীবদ্ধ করার জন্য পাঠ্য প্রম্পটে প্রদান করি(*playground_reference)। 

গ্রাহক অনুসন্ধান: হ্যালো, সম্প্রতি আমার ল্যাপটপ কীবোর্ডের একটি কী ভেঙে গেছে এবং আমাকে একটি প্রতিস্থাপন দরকার হবে:

শ্রেণীবদ্ধ বিভাগ:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## নতুন পণ্যের নাম তৈরি করুন
#### চ্যালেঞ্জ
উদাহরণ শব্দ থেকে পণ্যের নাম তৈরি করুন। এখানে আমরা প্রম্পটে পণ্যের সম্পর্কে তথ্য যোগ করি যার জন্য আমরা নাম তৈরি করতে যাচ্ছি। আমরা একটি অনুরূপ উদাহরণও প্রদান করি যাতে আমরা যে ধরনটি পেতে চাই তা স্পষ্ট হয়। আমরা তাপমাত্রার মানও বেশি রেখেছি যাতে র‍্যান্ডমনেস এবং আরও উদ্ভাবনী প্রতিক্রিয়া বৃদ্ধি পায়।

পণ্যের বিবরণ: একটি হোম মিল্কশেক মেকার
সিড শব্দ: দ্রুত, স্বাস্থ্যকর, কমপ্যাক্ট।
পণ্যের নাম: হোমশেকার, ফিট শেকার, কুইকশেক, শেক মেকার

পণ্যের বিবরণ: একটি জুতা যা যেকোনো পায়ের আকারের সাথে মানিয়ে যায়।
সিড শব্দ: অভিযোজিত, ফিট, ওম্নি-ফিট।


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## এমবেডিংস  
এই বিভাগে দেখানো হবে কীভাবে এমবেডিংস পুনরুদ্ধার করতে হয়, এবং শব্দ, বাক্য, ও ডকুমেন্টের মধ্যে সাদৃশ্য খুঁজে বের করতে হয়। নিম্নলিখিত নোটবুকগুলি চালানোর জন্য আপনাকে একটি মডেল ডিপ্লয় করতে হবে যা `text-embedding-ada-002` কে বেস মডেল হিসেবে ব্যবহার করে এবং .env ফাইলে তার ডিপ্লয়মেন্ট নাম সেট করতে হবে, `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` ভেরিয়েবল ব্যবহার করে।  


### মডেল শ্রেণীবিন্যাস - একটি এমবেডিং মডেল নির্বাচন করা  

**মডেল শ্রেণীবিন্যাস**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> টেক্সট-এমবেডিং  (এমবেডিং পরিবার)  
{capability} --> আডা             (অন্যান্য সকল এমবেডিং মডেল ২০২৪ সালে অবসান হবে)  
{input-type} --> প্রযোজ্য নয়    (শুধুমাত্র সার্চ মডেলের জন্য নির্দিষ্ট)  
{identifier} --> ০০২             (সংস্করণ ০০২)  

model = 'text-embedding-ada-002'  


  > [!NOTE] যদি এই নোটবুকটি Codespaces এ বা Devcontainer এর ভিতরে চালানো হয় তবে নিম্নলিখিত ধাপটি প্রয়োজনীয় নয়


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## CNN দৈনিক সংবাদ ডেটাসেট থেকে আর্টিকেল তুলনা
উৎস: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# রেফারেন্সসমূহ  
- [Microsoft Foundry - Azure OpenAI মডেলস](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry পোর্টাল](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# আরও সাহায্যের জন্য  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com)


# অবদানকারীরা
* লুইস লি  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
